Analyze customer reorder behaviour to identify repeat purchasing patterns, product and department loyalty, and how customer purchasing habits evolve over time.

In [3]:
overall_reorder = spark.sql("""
SELECT
    COUNT(*) AS items_sold,
    SUM(reordered) AS reordered_items,
    ROUND(AVG(reordered) * 100, 2) AS overall_reorder_rate
FROM order_products
""").toPandas()
overall_reorder.style.format({'items_sold': '{:,}',
'reordered_items': '{:,}','overall_reorder_rate': '{:.2f}'}).hide(axis='index')


StatementMeta(, 4e7b3296-5398-4066-b6d0-444bacfb6884, 5, Finished, Available, Finished, False)

items_sold,reordered_items,overall_reorder_rate
"33,819,106","19,955,360",59.01


_The dataset contains **33.82 million** purchased items, of which approximately **59%** were repeat purchases (based on the overall reorder rate).\
Nearly 6 out of every 10 purchased products were repeat purchases demonstrating strong customer loyalty toward frequently purchased items._

**What is the overall reorder rate, and how does it change across order sequence?**

In [5]:
order_sequence_analysis = spark.sql("""
SELECT
    CASE
        WHEN o.order_number BETWEEN 1 AND 10 THEN '1-10'
        WHEN o.order_number BETWEEN 11 AND 20 THEN '11-20'
        WHEN o.order_number BETWEEN 21 AND 30 THEN '21-30'
        WHEN o.order_number BETWEEN 31 AND 40 THEN '31-40'
        WHEN o.order_number BETWEEN 41 AND 50 THEN '41-50'
        WHEN o.order_number BETWEEN 51 AND 60 THEN '51-60'
        WHEN o.order_number BETWEEN 61 AND 70 THEN '61-70'
        WHEN o.order_number BETWEEN 71 AND 80 THEN '71-80'
        WHEN o.order_number BETWEEN 81 AND 90 THEN '81-90'
        ELSE '91+'
    END AS order_range,
    COUNT(op.product_id) AS total_items,
    ROUND(AVG(op.reordered) * 100, 1) AS reorder_rate,
    COUNT(DISTINCT o.order_id) AS orders
FROM orders o
JOIN order_products op ON o.order_id = op.order_id
GROUP BY order_range
ORDER BY order_range
""").toPandas()

order_sequence_analysis.style.format({'total_items': '{:,}',
'reorder_rate': '{:.1f}%','orders': '{:,}'}).hide(axis='index')

StatementMeta(, 4e7b3296-5398-4066-b6d0-444bacfb6884, 7, Finished, Available, Finished, False)

order_range,total_items,reorder_rate,orders
1-10,"16,474,179",42.4%,"1,642,411"
11-20,"7,398,009",69.2%,"727,252"
21-30,"4,002,535",75.8%,"389,585"
31-40,"2,406,832",79.0%,"231,516"
41-50,"1,473,736",81.1%,"142,400"
51-60,"858,737",82.1%,"85,529"
61-70,"515,437",82.8%,"53,194"
71-80,"331,918",83.7%,"35,119"
81-90,"216,171",84.6%,"23,504"
91+,"141,552",85.2%,"15,573"


_The consistent increase in reorder rates demonstrates that customer purchasing behaviour becomes progressively more habitual as shopping experience grows_

**Which department have the highest reorder rate among frequently purchased items?**

In [11]:
department_reorder = spark.sql("""
SELECT
    d.department,
    COUNT(*) AS total_items,
    ROUND(AVG(op.reordered) * 100, 1) AS reorder_rate
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY d.department
ORDER BY reorder_rate DESC
""").toPandas()

department_reorder.style.format({'total_items': '{:,}',
'reorder_rate': '{:.1f}%'}).hide(axis='index')

StatementMeta(, 4e7b3296-5398-4066-b6d0-444bacfb6884, 76, Finished, Available, Finished, False)

department,total_items,reorder_rate
dairy eggs,"5,631,067",67.0%
beverages,"2,804,175",65.4%
produce,"9,888,378",65.1%
bakery,"1,225,181",62.8%
deli,"1,095,540",60.8%
pets,"102,221",60.3%
babies,"438,743",57.8%
bulk,"35,932",57.7%
snacks,"3,006,412",57.4%
alcohol,"159,294",57.1%


_Departments combining high purchase volumes with reorder rates above 60% represent the strongest opportunities for inventory optimization, subscription services and personalized promotions_

**Which aisle have the highest reorder rate among frequently purchased items?**

In [12]:
aisle_reorder = spark.sql("""
SELECT
    a.aisle,
    d.department,
    COUNT(*) AS total_items,
    ROUND(AVG(op.reordered) * 100, 1) AS reorder_rate
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN aisles a ON p.aisle_id = a.aisle_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY a.aisle, d.department
HAVING COUNT(*) >= 1000
ORDER BY reorder_rate DESC
""").toPandas()

display(aisle_reorder.head(5).style.format({'total_items': '{:,}',
'reorder_rate': '{:.1f}%'}).hide(axis='index'))

display(aisle_reorder.tail(5).style.format({'total_items': '{:,}',
'reorder_rate': '{:.1f}%'}).hide(axis='index'))

StatementMeta(, 4e7b3296-5398-4066-b6d0-444bacfb6884, 89, Finished, Available, Finished, False)

aisle,department,total_items,reorder_rate
milk,dairy eggs,"923,659",78.2%
water seltzer sparkling water,beverages,"878,150",73.0%
fresh fruits,produce,"3,792,661",71.9%
eggs,dairy eggs,"472,009",70.6%
soy lactosefree,dairy eggs,"664,493",69.2%


aisle,department,total_items,reorder_rate
beauty,personal care,"6,455",21.3%
first aid,personal care,"11,411",19.6%
kitchen supplies,household,"9,620",19.5%
baking supplies decor,pantry,"24,786",16.8%
spices seasonings,pantry,"221,371",15.3%


_aisles including Milk, Fresh Fruits and Eggs, show the strongest repeat purchasing behaviour because customers buy these products frequently as part of their regular shopping routine. These aisles are ideal for personalized promotions and bundle recommendations. \
In contrast aisles with lower reorder rates such as Beauty, Kitchen Supplies and mostly spices may benefit more from seasonal campaigns and discounts_

**Which products have the highest reorder rate among frequently purchased items?**

In [13]:
product_reorder = spark.sql("""
SELECT
    p.product_name,
    d.department,
    a.aisle,
    COUNT(*) AS total_purchases,
    ROUND(AVG(op.reordered) * 100, 1) AS reorder_rate,
    SUM(op.reordered) AS times_reordered
FROM order_products op
JOIN products p ON op.product_id = p.product_id
JOIN aisles a ON p.aisle_id = a.aisle_id
JOIN departments d ON p.department_id = d.department_id
GROUP BY p.product_name, d.department, a.aisle
HAVING COUNT(*) >= 10000
ORDER BY reorder_rate DESC
""").toPandas()

display(product_reorder.head(5).style.format({'total_purchases': '{:,}',
'reorder_rate': '{:.1f}%','times_reordered': '{:,}'}).hide(axis='index'))

display(product_reorder.tail(5).style.format({'total_purchases': '{:,}',
'reorder_rate': '{:.1f}%','times_reordered': '{:,}'}).hide(axis='index'))

StatementMeta(, 4e7b3296-5398-4066-b6d0-444bacfb6884, 105, Finished, Available, Finished, False)

product_name,department,aisle,total_purchases,reorder_rate,times_reordered
"Milk, Organic, Vitamin D",dairy eggs,milk,"20,770",85.5%,"17,753"
Organic Reduced Fat Milk,dairy eggs,milk,"36,869",85.2%,"31,394"
Banana,produce,fresh fruits,"491,291",84.5%,"415,166"
Organic Whole Milk,dairy eggs,milk,"10,102",84.1%,"8,494"
Organic Lowfat 1% Milk,dairy eggs,milk,"15,352",84.1%,"12,914"


product_name,department,aisle,total_purchases,reorder_rate,times_reordered
Tomato Ketchup,pantry,condiments,"13,664",32.6%,"4,450"
Apple Cider Vinegar,pantry,oils vinegars,"10,143",32.1%,"3,255"
Light Brown Sugar,pantry,baking ingredients,"13,853",27.6%,"3,823"
Organic Dijon Mustard,pantry,condiments,"11,503",25.2%,"2,894"
Aluminum Foil,household,food storage,"11,181",21.9%,"2,446"


**What % of each customer's basket is new products vs repeat?**

In [24]:
customer_basket_segments = spark.sql("""
WITH customer_basket_type AS (
SELECT
    o.user_id,
    AVG(op.reordered) AS repeat_rate,
    1 - AVG(op.reordered) AS exploration_rate
FROM orders o
JOIN order_products op ON o.order_id = op.order_id
GROUP BY o.user_id
)
SELECT
CASE
    WHEN exploration_rate >= 0.75 THEN 'Explorer (75–100% new)'
    WHEN exploration_rate >= 0.50 THEN 'Mixed (50–75% new)'
    WHEN exploration_rate >= 0.25 THEN 'Habitual (25–50% new)'
ELSE 'Committed (<25% new)'
END AS customer_type,
COUNT(*) AS customers,
ROUND(AVG(exploration_rate) * 100, 1) AS avg_exploration,
ROUND(AVG(repeat_rate) * 100, 1) AS avg_repeat,
ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct_of_customers
FROM customer_basket_type
GROUP BY customer_type
ORDER BY avg_exploration DESC
""").toPandas()

customer_basket_segments.style.format({'customers': '{:,}',
'avg_exploration': '{:.1f}%', 'avg_repeat': '{:.1f}%',
'pct_of_customers': '{:.1f}%'}).hide(axis='index')

StatementMeta(, 4e7b3296-5398-4066-b6d0-444bacfb6884, 126, Finished, Available, Finished, False)

customer_type,customers,avg_exploration,avg_repeat,pct_of_customers
Explorer (75–100% new),"41,849",84.3%,15.7%,20.3%
Mixed (50–75% new),"81,888",61.8%,38.2%,39.7%
Habitual (25–50% new),"67,097",38.4%,61.6%,32.5%
Committed (<25% new),"15,375",19.0%,81.0%,7.5%


_Explorers benefit from product discovery and promotions, while Committed customers are ideal candidates for subscriptions_